In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🔵🟢 Blue-Green Deployment & Canary Release Patterns Guide

**Implementing safe deployment strategies with zero downtime**

This guide covers:
- Blue-green deployment architecture
- Canary releases and traffic splitting
- Health checks and monitoring
- Automated rollback
- Feature gates in deployments
- Progressive rollout strategies
- Deployment validation
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Blue-Green Deployment

### Deployment Controller
```python
from dataclasses import dataclass, field
from typing import Dict, Optional, List
from datetime import datetime
from enum import Enum

class DeploymentStatus(Enum):
    IDLE = "idle"
    DEPLOYING = "deploying"
    ACTIVE = "active"
    DRAINING = "draining"
    FAILED = "failed"

@dataclass
class Environment:
    """Deployment environment"""
    name: str  # "blue" or "green"
    version: str
    deployment_time: datetime
    instance_count: int
    is_active: bool
    status: DeploymentStatus = DeploymentStatus.IDLE
    metrics: Dict = field(default_factory=dict)

class BlueGreenController:
    """Manage blue-green deployments"""
    
    def __init__(self):
        self.blue: Optional[Environment] = None
        self.green: Optional[Environment] = None
        self.active_environment = "blue"
        self.deployment_history: List[Dict] = []
    
    def initialize(self, initial_version: str):
        """Initialize environments"""
        
        self.blue = Environment(
            name="blue",
            version=initial_version,
            deployment_time=datetime.utcnow(),
            instance_count=3,
            is_active=True,
            status=DeploymentStatus.ACTIVE
        )
        
        self.green = Environment(
            name="green",
            version=initial_version,
            deployment_time=datetime.utcnow(),
            instance_count=0,
            is_active=False,
            status=DeploymentStatus.IDLE
        )
    
    def deploy_new_version(self, new_version: str) -> bool:
        """Deploy new version to inactive environment"""
        
        # Determine which is inactive
        if self.active_environment == "blue":
            target_env = self.green
            other_env = self.blue
        else:
            target_env = self.blue
            other_env = self.green
        
        # Update inactive environment
        target_env.version = new_version
        target_env.deployment_time = datetime.utcnow()
        target_env.status = DeploymentStatus.DEPLOYING
        target_env.instance_count = other_env.instance_count
        
        # Record deployment
        self.deployment_history.append({
            'timestamp': datetime.utcnow().isoformat(),
            'from_version': other_env.version,
            'to_version': new_version,
            'target_environment': target_env.name,
            'status': 'started'
        })
        
        return True
    
    def validate_deployment(self, environment_name: str) -> bool:
        """Validate deployment health"""
        
        env = self.blue if environment_name == "blue" else self.green
        
        # Check instance health
        if env.instance_count == 0:
            return False
        
        # Check metrics
        error_rate = env.metrics.get('error_rate', 0.0)
        if error_rate > 0.05:  # >5% error rate
            return False
        
        # Check response time
        response_time = env.metrics.get('p99_response_time_ms', 0)
        if response_time > 1000:
            return False
        
        return True
    
    def switch_traffic(self, target_environment: str) -> bool:
        """Switch traffic to target environment"""
        
        if target_environment not in ["blue", "green"]:
            return False
        
        # Validate target
        env = self.blue if target_environment == "blue" else self.green
        if not self.validate_deployment(target_environment):
            print(f"Deployment validation failed for {target_environment}")
            return False
        
        # Update active environment
        self.active_environment = target_environment
        
        # Update status
        env.is_active = True
        env.status = DeploymentStatus.ACTIVE
        
        other_env = self.green if target_environment == "blue" else self.blue
        other_env.is_active = False
        other_env.status = DeploymentStatus.DRAINING
        
        # Record switch
        self.deployment_history[-1]['status'] = 'switched'
        self.deployment_history[-1]['switched_at'] = datetime.utcnow().isoformat()
        
        return True
    
    def rollback(self) -> bool:
        """Rollback to previous environment"""
        
        # Switch to non-active environment
        target = "blue" if self.active_environment == "green" else "green"
        
        result = self.switch_traffic(target)
        
        if result:
            self.deployment_history[-1]['status'] = 'rolled_back'
        
        return result
    
    def get_status(self) -> Dict:
        """Get deployment status"""
        
        return {
            'active_environment': self.active_environment,
            'blue': {
                'version': self.blue.version,
                'status': self.blue.status.value,
                'instances': self.blue.instance_count
            },
            'green': {
                'version': self.green.version,
                'status': self.green.status.value,
                'instances': self.green.instance_count
            }
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Canary Releases

### Canary Release Manager
```python
@dataclass
class CanaryConfig:
    """Canary release configuration"""
    initial_traffic_percentage: float = 5.0  # 5% to canary
    max_error_rate: float = 0.01  # 1% error threshold
    max_latency_ms: float = 500.0
    promotion_interval_seconds: int = 60
    max_duration_minutes: int = 30

class CanaryReleaseManager:
    """Manage canary releases"""
    
    def __init__(self, config: CanaryConfig = None):
        self.config = config or CanaryConfig()
        self.canary_active = False
        self.canary_traffic_percentage = 0.0
        self.canary_metrics: Dict = {}
        self.canary_start_time: Optional[datetime] = None
    
    def start_canary(self, new_version: str) -> bool:
        """Start canary release"""
        
        self.canary_active = True
        self.canary_traffic_percentage = self.config.initial_traffic_percentage
        self.canary_start_time = datetime.utcnow()
        
        self.canary_metrics = {
            'version': new_version,
            'error_count': 0,
            'request_count': 0,
            'total_latency_ms': 0.0
        }
        
        print(f"🔵 Canary started: {new_version} at {self.canary_traffic_percentage}% traffic")
        return True
    
    def promote_canary(self) -> bool:
        """Promote canary traffic"""
        
        if not self.canary_active:
            return False
        
        # Check metrics before promoting
        if not self._validate_canary_metrics():
            print("❌ Canary validation failed - not promoting")
            return False
        
        # Increase traffic
        new_traffic = min(100.0, self.canary_traffic_percentage + 25.0)
        self.canary_traffic_percentage = new_traffic
        
        print(f"✅ Canary promoted to {new_traffic}% traffic")
        
        if new_traffic >= 100.0:
            self._complete_canary()
        
        return True
    
    def abort_canary(self) -> bool:
        """Abort canary release"""
        
        if not self.canary_active:
            return False
        
        self.canary_active = False
        self.canary_traffic_percentage = 0.0
        
        print("❌ Canary aborted")
        return True
    
    def _validate_canary_metrics(self) -> bool:
        """Validate canary metrics"""
        
        if self.canary_metrics['request_count'] == 0:
            return False
        
        error_rate = self.canary_metrics['error_count'] / self.canary_metrics['request_count']
        if error_rate > self.config.max_error_rate:
            print(f"Error rate {error_rate} exceeds threshold {self.config.max_error_rate}")
            return False
        
        if self.canary_metrics['request_count'] > 0:
            avg_latency = self.canary_metrics['total_latency_ms'] / self.canary_metrics['request_count']
            if avg_latency > self.config.max_latency_ms:
                print(f"Avg latency {avg_latency}ms exceeds threshold {self.config.max_latency_ms}ms")
                return False
        
        return True
    
    def _complete_canary(self):
        """Complete canary release"""
        
        self.canary_active = False
        print("✅ Canary completed successfully - fully promoted")
    
    def record_request(self, is_error: bool, latency_ms: float):
        """Record canary request metrics"""
        
        self.canary_metrics['request_count'] += 1
        
        if is_error:
            self.canary_metrics['error_count'] += 1
        
        self.canary_metrics['total_latency_ms'] += latency_ms

class TrafficSplitter:
    """Split traffic between versions"""
    
    def __init__(self):
        self.version_traffic: Dict[str, float] = {}
    
    def configure_traffic_split(self, splits: Dict[str, float]):
        """Configure traffic split"""
        
        total = sum(splits.values())
        if abs(total - 100.0) > 0.01:
            return False
        
        self.version_traffic = splits
        return True
    
    def get_target_version(self, request_id: str) -> str:
        """Get target version for request"""
        
        import random
        
        # Use request ID for consistent routing
        random.seed(hash(request_id) % (2**32))
        rand = random.random() * 100
        
        cumulative = 0.0
        for version, percentage in self.version_traffic.items():
            cumulative += percentage
            if rand <= cumulative:
                return version
        
        return list(self.version_traffic.keys())[0]
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Health Checks & Rollback

### Health Monitoring
```python
class HealthChecker:
    """Monitor deployment health"""
    
    def __init__(self):
        self.health_checks: Dict[str, Dict] = {}
    
    def perform_health_check(self, environment: str) -> bool:
        """Perform comprehensive health check"""
        
        checks = {
            'connectivity': self._check_connectivity(environment),
            'database': self._check_database(environment),
            'dependencies': self._check_dependencies(environment),
            'disk_space': self._check_disk_space(environment),
            'cpu_usage': self._check_cpu_usage(environment)
        }
        
        self.health_checks[environment] = checks
        
        # All checks must pass
        return all(checks.values())
    
    def _check_connectivity(self, environment: str) -> bool:
        # Simulate connectivity check
        return True
    
    def _check_database(self, environment: str) -> bool:
        # Simulate database check
        return True
    
    def _check_dependencies(self, environment: str) -> bool:
        # Simulate dependency check
        return True
    
    def _check_disk_space(self, environment: str) -> bool:
        # Simulate disk space check
        return True
    
    def _check_cpu_usage(self, environment: str) -> bool:
        # Simulate CPU check
        return True

class AutomaticRollback:
    """Automatic rollback on issues"""
    
    def __init__(self, error_threshold: float = 0.05):
        self.error_threshold = error_threshold
        self.error_counts: Dict[str, int] = {}
        self.total_counts: Dict[str, int] = {}
    
    def should_rollback(self, environment: str) -> bool:
        """Determine if rollback needed"""
        
        if environment not in self.total_counts:
            return False
        
        total = self.total_counts[environment]
        errors = self.error_counts.get(environment, 0)
        
        if total == 0:
            return False
        
        error_rate = errors / total
        return error_rate > self.error_threshold
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Deployment Checklist

✅ **Blue-Green Setup**
- [ ] Two environments configured
- [ ] Load balancer switching
- [ ] Deployment automation
- [ ] Version tracking
- [ ] History logging

✅ **Canary Release**
- [ ] Initial traffic configured (e.g., 5%)
- [ ] Metrics collection
- [ ] Promotion strategy
- [ ] Abort capability
- [ ] Timeline tracking

✅ **Health & Safety**
- [ ] Health checks defined
- [ ] Validation gates
- [ ] Rollback automation
- [ ] Error rate monitoring
- [ ] Performance baselines

✅ **Operational**
- [ ] Documentation
- [ ] Team runbooks
- [ ] Monitoring dashboard
- [ ] Alert configuration
- [ ] Communication plan

✅ **Validation**
- [ ] Smoke tests
- [ ] Integration tests
- [ ] Performance tests
- [ ] Regression tests
- [ ] Load testing
</VSCode.Cell>
```